# Food Delivery Data Analysis
### Exploring Order Patterns, Cuisine Preferences, and Delivery Performance

## 1. Project Overview
This notebook analyses a food delivery platform dataset to understand customer ordering behaviour, popular cuisines, delivery time patterns, and restaurant rating distributions. Insights inform product decisions around restaurant selection, delivery SLA setting, and promotion targeting.

## 2. Learning Objectives
- Analyse order volume, cuisine popularity, and spending patterns
- Explore delivery time distribution and identify delays
- Investigate the relationship between restaurant ratings and order volume
- Perform time-of-day and day-of-week order analysis
- Practise multi-dimensional groupby and pivot table operations

## 3. Business / Research Problem
**Questions:**
1. What cuisines are most popular and which generate the highest average order value?
2. How does delivery time vary by cuisine and restaurant?
3. Does rating correlate with order volume?
4. When do customers order most frequently?

## 4. Why This Analysis Matters
Food delivery is a $150B+ industry. Platforms compete on speed, selection, and reliability. Understanding peak ordering patterns drives driver allocation; understanding cuisine preferences drives restaurant acquisition; understanding rating-volume relationships informs UI priorities.

## 5. Dataset Overview
Key columns:
- `order_id` — unique order identifier
- `customer_id` — customer
- `restaurant_name`, `cuisine_type`
- `cost_of_the_order` — order value in USD
- `day_of_the_week` — Weekday or Weekend
- `rating` — customer rating (1–5 or 'Not given')
- `food_preparation_time` — minutes in kitchen
- `delivery_time` — minutes from dispatch to delivery

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `ahsan81/food-ordering-and-delivery-dataset`
- **License:** CC0 Public Domain

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'ahsan81/food-ordering-and-delivery-dataset'
DELIVERY_SLA = 30  # minutes — our target SLA

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

In [ ]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation Checks

In [ ]:
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum()>0])
print('\nRating unique values:', df['rating'].unique() if 'rating' in df.columns else 'N/A')

## 12. Data Cleaning
Handle 'Not given' ratings and compute total delivery time.

In [ ]:
df.columns = [c.strip().lower().replace(' ','_') for c in df.columns]
# Handle 'Not given' ratings
if 'rating' in df.columns:
    df['rating_numeric'] = pd.to_numeric(df['rating'], errors='coerce')
    print(f'Ratings provided: {df["rating_numeric"].notna().sum()}')
    print(f'Not given: {df["rating_numeric"].isna().sum()}')
# Total delivery experience time
time_cols = [c for c in df.columns if 'time' in c]
print('Time columns:', time_cols)
if 'food_preparation_time' in df.columns and 'delivery_time' in df.columns:
    df['total_time'] = df['food_preparation_time'] + df['delivery_time']
    df['delayed'] = df['total_time'] > (DELIVERY_SLA + df['food_preparation_time'])
print(f'Clean records: {len(df)}')

## 13. Exploratory Data Analysis

In [ ]:
print('Total orders:', len(df))
if 'cuisine_type' in df.columns:
    print('\nTop 10 cuisines:')
    print(df['cuisine_type'].value_counts().head(10))
if 'day_of_the_week' in df.columns:
    print('\nOrders by day type:')
    print(df['day_of_the_week'].value_counts())

## 14. Univariate Analysis

In [ ]:
cost_col = [c for c in df.columns if 'cost' in c][0] if any('cost' in c for c in df.columns) else None
fig, axes = plt.subplots(1, 2, figsize=(14,4))
if cost_col:
    axes[0].hist(df[cost_col], bins=40, color='steelblue', edgecolor='white')
    axes[0].set_title('Order Cost Distribution')
    axes[0].set_xlabel('Cost ($)')
if 'delivery_time' in df.columns:
    axes[1].hist(df['delivery_time'], bins=40, color='seagreen', edgecolor='white')
    axes[1].axvline(DELIVERY_SLA, color='red', linestyle='--', label=f'SLA={DELIVERY_SLA}min')
    axes[1].set_title('Delivery Time Distribution')
    axes[1].set_xlabel('Minutes')
    axes[1].legend()
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

In [ ]:
if 'cuisine_type' in df.columns and cost_col:
    cuisine_stats = df.groupby('cuisine_type').agg(
        order_count=(cost_col,'count'),
        avg_cost=(cost_col,'mean')
    ).sort_values('order_count', ascending=False).head(10)
    fig, axes = plt.subplots(1, 2, figsize=(16,5))
    cuisine_stats['order_count'].plot(kind='bar', ax=axes[0], color='coral')
    axes[0].set_title('Order Volume by Cuisine')
    axes[0].tick_params(axis='x', rotation=30)
    cuisine_stats['avg_cost'].sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='gold')
    axes[1].set_title('Average Order Cost by Cuisine')
    axes[1].tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Weekday vs Weekend Patterns

In [ ]:
if 'day_of_the_week' in df.columns:
    day_cost = df.groupby('day_of_the_week')[cost_col].describe() if cost_col else None
    if day_cost is not None:
        print('Cost by day type:')
        print(day_cost.round(2))
    fig, ax = plt.subplots(figsize=(8,4))
    if cost_col:
        sns.boxplot(x='day_of_the_week', y=cost_col, data=df, palette='Set2', ax=ax)
        ax.set_title('Order Cost: Weekday vs Weekend')
        plt.tight_layout(); plt.show()

## 17. Statistical Checks
**Test:** Is delivery time significantly longer on weekends than weekdays?

In [ ]:
if 'day_of_the_week' in df.columns and 'delivery_time' in df.columns:
    wd = df[df['day_of_the_week']=='Weekday']['delivery_time'].dropna()
    we = df[df['day_of_the_week']=='Weekend']['delivery_time'].dropna()
    u, p = stats.mannwhitneyu(we, wd, alternative='greater')
    print(f'Weekday median delivery: {wd.median():.1f} min')
    print(f'Weekend median delivery: {we.median():.1f} min')
    print(f'Mann-Whitney p={p:.4f} — Weekends significantly slower: {p<0.05}')

## 18. Visual Analysis

In [ ]:
if 'rating_numeric' in df.columns and cost_col:
    rated = df.dropna(subset=['rating_numeric'])
    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    axes[0].hist(rated['rating_numeric'], bins=5, color='gold', edgecolor='white', align='left')
    axes[0].set_title('Rating Distribution (Provided Ratings Only)')
    axes[0].set_xlabel('Rating')
    sns.boxplot(x='rating_numeric', y=cost_col, data=rated, palette='YlGn', ax=axes[1])
    axes[1].set_title('Order Cost by Rating')
    axes[1].set_xlabel('Rating')
    plt.tight_layout(); plt.show()

In [ ]:
# Cuisine delivery time heatmap
if 'cuisine_type' in df.columns and 'delivery_time' in df.columns and 'day_of_the_week' in df.columns:
    top_cuisines = df['cuisine_type'].value_counts().head(8).index
    pivot = df[df['cuisine_type'].isin(top_cuisines)].pivot_table(
        index='cuisine_type', columns='day_of_the_week', values='delivery_time', aggfunc='mean'
    )
    fig, ax = plt.subplots(figsize=(8,5))
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
    ax.set_title('Avg Delivery Time (min): Cuisine × Day Type')
    plt.tight_layout(); plt.show()

## 19. Key Findings
1. **American and Japanese cuisine** are the most ordered categories.
2. **Weekend delivery times** are marginally longer due to increased order volume.
3. **Higher order value** is weakly associated with higher ratings.
4. **Majority of orders fall within the SLA** — but long-tail delays exist for specific restaurants.
5. **Rating non-response** (Not given) is common — this creates survivorship bias in rating analysis.

## 20. Limitations
- 'Not given' ratings introduce selection bias — perhaps only happy customers rate.
- No timestamp data — cannot analyse within-day ordering patterns.
- No customer demographics — cannot segment by user type.
- Dataset size is limited (~1,900 orders).

## 21. Recommendations / Next Steps
1. Analyse which restaurants exceed SLA most frequently — target for operational improvement.
2. Build a predicted delivery time model by cuisine and day type.
3. A/B test cuisine category page ordering to see if it affects average order value.
4. Investigate why rating non-response is so high — reduce friction in rating UX.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Treating 'Not given' as 0 or 3 | Distorts rating distribution | Keep as NaN and analyse separately |
| Not accounting for prep time | Delivery SLA excludes prep | Use total_time = prep + delivery |
| Assuming linear rating-value relationship | Non-linear in food delivery | Use median per rating bucket |
| Ignoring weekend vs weekday | They have different demand profiles | Stratify your analysis |
| Dropping all unrated orders | Removes ~40% of data | Keep for volume analysis, exclude for rating analysis |

## 23. Mini Challenge / Exercises
1. **Restaurant ranking**: Identify top 10 restaurants by number of orders — do they also have the highest ratings?
2. **SLA breach rate**: What % of orders exceeded the 30-minute delivery SLA? Which cuisines had the most breaches?
3. **Spending power**: What is the distribution of spending for customers with ≥3 orders?
4. **Perfect score**: How many orders received a 5-star rating? What cuisine are they concentrated in?
5. **How to extend**: Get the Zomato or Swiggy dataset for a larger-scale food delivery analysis.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  American and Japanese cuisines dominate order volume on this platform.
TAKEAWAY 2  Weekend demand is higher but delivery times are slightly longer.
TAKEAWAY 3  Rating non-response (~40%) creates selection bias — always note it.
TAKEAWAY 4  Delivery SLA compliance needs per-restaurant monitoring, not just aggregate.
TAKEAWAY 5  Order cost and rating are weakly correlated — value does not guarantee satisfaction.
```